In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator
load_dotenv()

True

In [8]:
model = ChatMistralAI(model="mistral-small-2506")

In [9]:
# baseMessage means here can be any type like ai,human,system message and we use add_message when we dont want 
# previous message to overwrite the previous one used when there are messages not simple str
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [10]:
def chat_node(state: ChatState) -> ChatState:
    messages = state['messages']
    response = model.invoke(messages)
    return {"messages": [response]}

In [11]:
# here the problem is it will not remember old chat because when we ask any ques .invoke gets called and due to it
# every time new state starts previous one gets erased so no we have to same the prev state somewhere in RAM
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

# result = chatbot.invoke({
#     "messages": [HumanMessage(content="what is capital of india?")]
# })
# print(result['messages'][-1].content)

In [12]:
thread_id = "1"
while True:
    prompt =  input("YOU: ")
    if prompt.strip().lower() == "exit":
        break
    config = {"configurable": {"thread_id": thread_id}}
    response = chatbot.invoke({
        "messages": [HumanMessage(content=prompt)]
    }, config=config)
    print("AI: ", response["messages"][-1].content)